In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.death;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_tw.death;

In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.death
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_death AS
SELECT
  source_to_person.person_id AS person_id,
  CAST(dbo_person_other.dateofdeath AS DATE) AS death_date,
  CAST(dbo_person_other.dateofdeath AS TIMESTAMP) AS death_datetime,
  32817 death_type_concept_id,
  NULL cause_concept_id,
  NULL AS cause_source_value,
  NULL AS cause_source_concept_id,
  'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person_other
ON dbo_person_other.id = dbo_person.id
LEFT JOIN _exponent.omop_mapping.source_to_person
ON source_to_person.person_source_value = CONCAT_WS(CHR(31), 'allscripts_tw','dbo_person', 'id', dbo_person.id)
WHERE 1=1
AND dbo_person.id IS NOT NULL
AND dbo_person_other.dateofdeath IS NOT NULL
AND source_to_person.person_id IS NOT NULL


In [0]:
%sql
MERGE INTO _exponent.omop_silver.death AS target
USING silver_death AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
     target.death_date              <=> source.death_date
 AND target.death_datetime          <=> source.death_datetime
 AND target.death_type_concept_id   <=> source.death_type_concept_id
 AND target.cause_concept_id        <=> source.cause_concept_id
 AND target.cause_source_value      <=> source.cause_source_value
 AND target.cause_source_concept_id <=> source.cause_source_concept_id
) THEN UPDATE SET
  target.death_date              = source.death_date,
  target.death_datetime          = source.death_datetime,
  target.death_type_concept_id   = source.death_type_concept_id,
  target.cause_concept_id        = source.cause_concept_id,
  target.cause_source_value      = source.cause_source_value,
  target.cause_source_concept_id = source.cause_source_concept_id,
  target.source_system           = source.source_system,
  target.last_mod_tsp            = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  person_id,
  death_date,
  death_datetime,
  death_type_concept_id,
  cause_concept_id,
  cause_source_value,
  cause_source_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.person_id,
  source.death_date,
  source.death_datetime,
  source.death_type_concept_id,
  source.cause_concept_id,
  source.cause_source_value,
  source.cause_source_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold AS
SELECT
  death.person_id,
  death.death_date,
  death.death_datetime,
  COALESCE(death.death_type_concept_id, 0) AS death_type_concept_id,
  death.cause_concept_id,
  death.cause_source_value,
  death.cause_source_concept_id
FROM _exponent.omop_silver.death death
INNER JOIN _exponent.omop_tw.person person
  ON death.person_id = person.person_id
WHERE death.person_id IS NOT NULL
  AND death.death_date IS NOT NULL
  AND death.death_date >= '1950-01-01'
  AND death.source_system = 'allscripts_tw';

In [0]:
%sql
MERGE INTO _exponent.omop_tw.death AS target
USING gold AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
     target.death_date              <=> source.death_date
 AND target.death_datetime          <=> source.death_datetime
 AND target.death_type_concept_id   <=> source.death_type_concept_id
 AND target.cause_concept_id        <=> source.cause_concept_id
 AND target.cause_source_value      <=> source.cause_source_value
 AND target.cause_source_concept_id <=> source.cause_source_concept_id
) THEN UPDATE SET
  target.death_date              = source.death_date,
  target.death_datetime          = source.death_datetime,
  target.death_type_concept_id   = source.death_type_concept_id,
  target.cause_concept_id        = source.cause_concept_id,
  target.cause_source_value      = source.cause_source_value,
  target.cause_source_concept_id = source.cause_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  death_date,
  death_datetime,
  death_type_concept_id,
  cause_concept_id,
  cause_source_value,
  cause_source_concept_id
) VALUES (
  source.person_id,
  source.death_date,
  source.death_datetime,
  source.death_type_concept_id,
  source.cause_concept_id,
  source.cause_source_value,
  source.cause_source_concept_id
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.death AS target
USING gold AS source
ON target.person_id = source.person_id

WHEN MATCHED AND NOT (
     target.death_date              <=> source.death_date
 AND target.death_datetime          <=> source.death_datetime
 AND target.death_type_concept_id   <=> source.death_type_concept_id
 AND target.cause_concept_id        <=> source.cause_concept_id
 AND target.cause_source_value      <=> source.cause_source_value
 AND target.cause_source_concept_id <=> source.cause_source_concept_id
) THEN UPDATE SET
  target.death_date              = source.death_date,
  target.death_datetime          = source.death_datetime,
  target.death_type_concept_id   = source.death_type_concept_id,
  target.cause_concept_id        = source.cause_concept_id,
  target.cause_source_value      = source.cause_source_value,
  target.cause_source_concept_id = source.cause_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  death_date,
  death_datetime,
  death_type_concept_id,
  cause_concept_id,
  cause_source_value,
  cause_source_concept_id
) VALUES (
  source.person_id,
  source.death_date,
  source.death_datetime,
  source.death_type_concept_id,
  source.cause_concept_id,
  source.cause_source_value,
  source.cause_source_concept_id
);